# Multi-Label CNN-LSTM for Exercise Form Analysis (REFACTORED)

**VERSION**: 06 - Multi-Label Implementation  
**STATUS**: 🚀 Production-ready multi-label classification system  
**ARCHITECTURE**: CNN-LSTM with user-level splits and concatenation fusion  

## 🎯 Refactoring Goals ACHIEVED

### ✅ Key Changes from Original
1. **User-level splits**: No data leakage between train/val/test
2. **Multi-label targets**: `[y_good, y_posture, y_depth]` format
3. **Rep-level classification**: One prediction per video
4. **Concatenation fusion**: `sequence_repr + biomech_proj → fused_features`
5. **BCEWithLogitsLoss**: Multi-label loss function
6. **Per-head metrics**: Precision/Recall/F1 for each head
7. **Threshold optimization**: Production-ready decision boundaries

### 📈 Expected Performance Improvements
- **Safer evaluation**: User-level splits eliminate optimistic bias
- **Better decisions**: Per-head thresholds optimize each fault type
- **Future proof**: Multi-label design supports complex fault combinations
- **Production ready**: Clear decision boundaries for deployment

## 🔧 Architecture Overview
```
Input: Pose sequences (T, 33, 3) + Optional 7D biomech features
↓
Temporal CNN → BiLSTM → Multi-head Attention → sequence_repr (512)
↓
Concatenation Fusion: [sequence_repr, biomech_proj] → (576)
↓
Fusion Head → Multi-label outputs (3 independent logits)
↓
BCEWithLogitsLoss + Threshold Optimization
```

In [13]:
# Cell 1: Imports and Setup for Multi-Label System
import os
import sys
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import logging
from pathlib import Path
from datetime import datetime
from collections import Counter, defaultdict
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

# PyTorch imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence, pad_sequence
from torch.optim.lr_scheduler import ReduceLROnPlateau

# Sklearn imports
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, precision_recall_curve
)
from sklearn.linear_model import LogisticRegression

# Set up environment
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 MULTI-LABEL FORMIQ REFACTORING SYSTEM")
print(f"Device: {device}")
print(f"PyTorch version: {torch.__version__}")

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Setup plotting
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_theme(style="whitegrid")
%matplotlib inline

print("✅ Environment setup complete for multi-label system")

🚀 MULTI-LABEL FORMIQ REFACTORING SYSTEM
Device: cpu
PyTorch version: 2.2.0
✅ Environment setup complete for multi-label system


In [14]:
# Cell 2: User-Level Data Splits (No Data Leakage)
# Implements user-level train/val/test splits and multi-label target conversion

def extract_user_id(video_name: str, source_dataset: str) -> str:
    """Extract user ID from video name based on dataset source"""
    if source_dataset == "squat_more":
        # Pattern: userID_repNumber (e.g., "49437_3" → user_id = "49437")
        return video_name.split('_')[0]
    elif source_dataset == "formiq":
        # Pattern: videoID_good_rep_X or plain numbers
        if '_' in video_name:
            base_id = video_name.split('_')[0]
            return f"formiq_{base_id}"
        else:
            return f"formiq_{video_name}"
    else:
        return f"unknown_{video_name}"

def create_multilabel_targets(class_label: str) -> list:
    """Convert 3-class label to multi-label format [y_good, y_posture, y_depth]"""
    y_good = 1 if class_label == 'good_form' else 0
    y_posture = 1 if class_label == 'posture_fault' else 0  
    y_depth = 1 if class_label == 'depth_fault' else 0
    return [y_good, y_posture, y_depth]

def create_user_level_multilabel_splits():
    """Create user-level splits with multi-label targets"""
    
    print("🔄 Creating user-level splits with multi-label targets...")
    
    # Load processing summary
    processing_summary_path = "data/squat_processed/processing_summary_balanced_3class.json"
    with open(processing_summary_path, 'r') as f:
        videos_data = json.load(f)
    
    print(f"✅ Loaded {len(videos_data)} videos from dataset")
    
    # Group videos by user ID
    user_to_videos = defaultdict(list)
    user_to_classes = defaultdict(set)
    
    for video_data in videos_data:
        video_name = video_data['video_name']
        source_dataset = video_data.get('source_dataset', 'unknown')
        class_label = video_data['enhanced_labels']['video_level']['class_label']
        
        user_id = extract_user_id(video_name, source_dataset)
        user_to_videos[user_id].append({
            'video_name': video_name,
            'class_label': class_label,
            'video_data': video_data
        })
        user_to_classes[user_id].add(class_label)
    
    print(f"📊 User-level grouping results:")
    print(f"   Total unique users: {len(user_to_videos)}")
    print(f"   Videos per user: {np.mean([len(videos) for videos in user_to_videos.values()]):.1f} ± {np.std([len(videos) for videos in user_to_videos.values()]):.1f}")
    
    # Create user-level class labels for stratification
    user_ids = list(user_to_videos.keys())
    user_primary_classes = []
    
    for user_id in user_ids:
        user_classes = [video['class_label'] for video in user_to_videos[user_id]]
        primary_class = Counter(user_classes).most_common(1)[0][0]
        user_primary_classes.append(primary_class)
    
    # Stratified user-level splitting
    user_ids = np.array(user_ids)
    user_primary_classes = np.array(user_primary_classes)
    
    # Split users (not videos) into train/temp
    user_train, user_temp, _, class_temp = train_test_split(
        user_ids, user_primary_classes,
        test_size=0.30,  # 70% train, 30% val+test
        stratify=user_primary_classes,
        random_state=42
    )
    
    # Split temp users into val/test
    user_val, user_test, _, _ = train_test_split(
        user_temp, class_temp,
        test_size=0.50,  # 15% val, 15% test
        stratify=class_temp,
        random_state=42
    )
    
    print(f"✅ User-level splits created:")
    print(f"   Train users: {len(user_train)}")
    print(f"   Validation users: {len(user_val)}")
    print(f"   Test users: {len(user_test)}")
    
    # Map users back to videos and create multi-label targets
    splits = {}
    split_names = ['train', 'validation', 'test']
    split_users = [user_train, user_val, user_test]
    
    for split_name, split_user_list in zip(split_names, split_users):
        split_videos = []
        split_classes = []
        split_multilabel_targets = []
        split_video_data = []
        
        for user_id in split_user_list:
            for video_info in user_to_videos[user_id]:
                split_videos.append(video_info['video_name'])
                split_classes.append(video_info['class_label'])
                split_multilabel_targets.append(create_multilabel_targets(video_info['class_label']))
                split_video_data.append(video_info['video_data'])
        
        splits[split_name] = {
            'user_ids': split_user_list.tolist(),
            'video_names': split_videos,
            'class_labels': split_classes,
            'multilabel_targets': split_multilabel_targets,
            'video_data': split_video_data,
            'size': len(split_videos),
            'num_users': len(split_user_list)
        }
    
    # Validate no user leakage
    train_users = set(splits['train']['user_ids'])
    val_users = set(splits['validation']['user_ids'])
    test_users = set(splits['test']['user_ids'])
    
    assert len(train_users.intersection(val_users)) == 0, "Train/Val user leakage!"
    assert len(train_users.intersection(test_users)) == 0, "Train/Test user leakage!"
    assert len(val_users.intersection(test_users)) == 0, "Val/Test user leakage!"
    
    print(f"✅ NO USER LEAKAGE - splits are clean")
    
    # Save splits
    output_data = {
        'metadata': {
            'creation_date': pd.Timestamp.now().isoformat(),
            'total_videos': sum(split_data['size'] for split_data in splits.values()),
            'total_users': sum(split_data['num_users'] for split_data in splits.values()),
            'class_names': ['good_form', 'posture_fault', 'depth_fault'],
            'multilabel_encoding': {
                'y_good': 'good_form vs not good',
                'y_posture': 'posture_fault yes/no', 
                'y_depth': 'depth_fault yes/no'
            },
            'user_level_splits': True
        },
        'splits': splits
    }
    
    output_path = "data/squat_processed/user_level_multilabel_splits.json"
    with open(output_path, 'w') as f:
        json.dump(output_data, f, indent=2)
    
    print(f"💾 Saved user-level splits: {output_path}")
    
    # Print summary
    for split_name, split_data in splits.items():
        multilabel_array = np.array(split_data['multilabel_targets'])
        print(f"\n📊 {split_name.upper()} SET:")
        print(f"   Videos: {split_data['size']} | Users: {split_data['num_users']}")
        print(f"   Multi-label targets:")
        print(f"     Good form: {np.sum(multilabel_array[:, 0])} positive")
        print(f"     Posture fault: {np.sum(multilabel_array[:, 1])} positive")
        print(f"     Depth fault: {np.sum(multilabel_array[:, 2])} positive")
    
    return splits

# Execute user-level splitting
user_level_splits = create_user_level_multilabel_splits()
print("\n🎉 USER-LEVEL MULTILABEL SPLITS COMPLETE!")

🔄 Creating user-level splits with multi-label targets...
✅ Loaded 1625 videos from dataset
📊 User-level grouping results:
   Total unique users: 1625
   Videos per user: 1.0 ± 0.0
✅ User-level splits created:
   Train users: 1137
   Validation users: 244
   Test users: 244
✅ NO USER LEAKAGE - splits are clean
💾 Saved user-level splits: data/squat_processed/user_level_multilabel_splits.json

📊 TRAIN SET:
   Videos: 1137 | Users: 1137
   Multi-label targets:
     Good form: 404 positive
     Posture fault: 405 positive
     Depth fault: 328 positive

📊 VALIDATION SET:
   Videos: 244 | Users: 244
   Multi-label targets:
     Good form: 87 positive
     Posture fault: 87 positive
     Depth fault: 70 positive

📊 TEST SET:
   Videos: 244 | Users: 244
   Multi-label targets:
     Good form: 87 positive
     Posture fault: 86 positive
     Depth fault: 71 positive

🎉 USER-LEVEL MULTILABEL SPLITS COMPLETE!


In [15]:
# UPDATED CELL 3: Multi-Label Dataset with Integrated Depth Features
# REPLACE the existing Cell 3 in your main notebook with this code

# Import depth processing modules
import sys
sys.path.append('src')
from cleaned_biomech_extractor import create_cleaned_biomech_extractor
from depth_rules import (
    load_depth_thresholds, normalize_landmarks_pelvis_shoulder,
    compute_depth_signals, predict_bad_depth, predict_good_depth,
    signals_to_tensor
)

class EnhancedMultiLabelSquatDataset(Dataset):
    """Enhanced Multi-label PyTorch Dataset with integrated depth features"""
    
    def __init__(self, video_names, multilabel_targets, max_sequence_length=300, 
                 enable_depth_integration=True, enable_clean_biomech=True):
        self.video_names = video_names
        self.multilabel_targets = multilabel_targets
        self.max_sequence_length = max_sequence_length
        self.enable_depth_integration = enable_depth_integration
        self.enable_clean_biomech = enable_clean_biomech
        
        # Load frame labels data
        frame_labels_path = "data/squat_processed/balanced_3class_frame_labels.json"
        with open(frame_labels_path, 'r') as f:
            self.frame_labels_data = json.load(f)
        
        # Initialize depth processing (with fallbacks)
        if self.enable_depth_integration:
            try:
                self.depth_config = load_depth_thresholds("data/squat_processed/depth_thresholds.json")
                print(f"✅ Loaded depth thresholds: {self.depth_config['metadata']['best_combination']['name']}")
            except Exception as e:
                print(f"⚠️  Failed to load depth config: {e}, using fallback")
                self.depth_config = None
        else:
            self.depth_config = None
        
        # Initialize clean biomech extractor (with fallback)
        if self.enable_clean_biomech:
            try:
                self.biomech_extractor = create_cleaned_biomech_extractor(confidence_threshold=0.5)
                print(f"✅ Loaded cleaned 5D biomechanical feature extractor")
            except Exception as e:
                print(f"⚠️  Failed to load biomech extractor: {e}, using fallback")
                self.biomech_extractor = None
        else:
            self.biomech_extractor = None
        
        # Filter valid videos
        self.valid_videos = []
        self.valid_targets = []
        
        for i, video_name in enumerate(video_names):
            if self._validate_video(video_name):
                self.valid_videos.append(video_name)
                self.valid_targets.append(multilabel_targets[i])
        
        print(f"Dataset: {len(self.valid_videos)}/{len(video_names)} valid videos")
        if self.enable_depth_integration:
            print(f"   Depth integration: {'ENABLED' if self.depth_config else 'FALLBACK'}")
        if self.enable_clean_biomech:
            print(f"   Clean biomech: {'ENABLED' if self.biomech_extractor else 'FALLBACK'}")
    
    def _validate_video(self, video_name):
        """Validate that video has required data"""
        if video_name not in self.frame_labels_data['videos']:
            return False
        
        video_data = self.frame_labels_data['videos'][video_name]
        keypoints_path = video_data.get('keypoints_path', '')
        
        if not keypoints_path or not Path(keypoints_path).exists():
            return False
        
        frame_count = video_data.get('frame_count', 0)
        return 30 <= frame_count <= self.max_sequence_length
    
    def __len__(self):
        return len(self.valid_videos)
    
    def __getitem__(self, idx):
        video_name = self.valid_videos[idx]
        multilabel_target = self.valid_targets[idx]
        
        video_data = self.frame_labels_data['videos'][video_name]
        
        # Load keypoints
        keypoints_path = video_data['keypoints_path']
        with open(keypoints_path, 'r') as f:
            keypoints_data = json.load(f)
        
        # Process keypoints to standardized format
        keypoints = self._process_keypoints(keypoints_data)
        
        # Truncate if necessary
        if len(keypoints) > self.max_sequence_length:
            keypoints = keypoints[:self.max_sequence_length]
        
        # EXTRACT INTEGRATED FEATURES (9D total)
        rep_features = self._extract_integrated_features(keypoints, video_name)
        
        return {
            'video_name': video_name,
            'keypoints': torch.FloatTensor(keypoints),  # (T, 33, 3)
            'multilabel_target': torch.FloatTensor(multilabel_target),  # (3,)
            'rep_features': torch.FloatTensor(rep_features),  # (9,) - ENHANCED!
            'sequence_length': torch.LongTensor([len(keypoints)])
        }
    
    def _extract_integrated_features(self, keypoints: np.ndarray, video_name: str) -> np.ndarray:
        """
        Extract 9D integrated features: [5D_clean_biomech + 2D_depth_signals + 2D_depth_rules]
        
        Args:
            keypoints: Shape (T, 33, 3) raw keypoints
            video_name: Video identifier for caching/logging
            
        Returns:
            9D feature vector
        """
        try:
            # 1. NORMALIZE KEYPOINTS (same as calibration)
            if self.enable_depth_integration:
                normalized_kp = normalize_landmarks_pelvis_shoulder(keypoints, confidence_threshold=0.5)
            else:
                # Fallback: use raw coordinates for biomech only
                normalized_kp = keypoints[:, :, :2]  # Just take x,y
            
            # 2. EXTRACT 5D CLEAN BIOMECHANICAL FEATURES
            if self.enable_clean_biomech and self.biomech_extractor:
                biomech_features = self.biomech_extractor.extract_rep_features(keypoints)
                if len(biomech_features) != 5:
                    raise ValueError(f"Expected 5D biomech features, got {len(biomech_features)}D")
            else:
                # Fallback: stable default biomech features
                biomech_features = np.array([90.0, 45.0, 90.0, 5.0, 0.8])  # [min_knee, knee_rom, min_hip, torso_lean, torso_stability]
            
            # 3. EXTRACT 2D DEPTH SIGNALS
            if self.enable_depth_integration and normalized_kp is not None:
                depth_signals = compute_depth_signals(normalized_kp, bottom_frac=0.10)
                depth_signal_tensor = signals_to_tensor(depth_signals)  # [hip_knee_y_bottom, knee_angle_bottom]
                
                # 4. EXTRACT 2D DEPTH RULES
                if self.depth_config:
                    bad_depth = predict_bad_depth(depth_signals, self.depth_config)
                    good_depth = predict_good_depth(depth_signals, self.depth_config)
                else:
                    # Fallback rules
                    bad_depth = 1 if depth_signals['hip_knee_y_bottom'] <= -100 else 0
                    good_depth = 1 - bad_depth
                
                depth_rules = np.array([float(bad_depth), float(good_depth)], dtype=np.float32)
            else:
                # Fallback: neutral depth features
                depth_signal_tensor = np.array([0.0, 90.0], dtype=np.float32)
                depth_rules = np.array([0.0, 1.0], dtype=np.float32)  # Assume good depth by default
            
            # 5. CONCATENATE ALL FEATURES: [5D + 2D + 2D] = 9D
            integrated_features = np.concatenate([
                biomech_features.astype(np.float32),  # 5D clean biomech
                depth_signal_tensor,                   # 2D depth signals
                depth_rules                            # 2D depth rules
            ])
            
            # Validate output shape
            if len(integrated_features) != 9:
                raise ValueError(f"Expected 9D integrated features, got {len(integrated_features)}D")
            
            # Ensure no NaN/inf values
            integrated_features = np.nan_to_num(integrated_features, nan=0.0, posinf=1e6, neginf=-1e6)
            
            return integrated_features
            
        except Exception as e:
            # Ultimate fallback: return safe 9D vector
            print(f"⚠️  Feature extraction failed for {video_name}: {e}")
            return np.array([90.0, 45.0, 90.0, 5.0, 0.8, 0.0, 90.0, 0.0, 1.0], dtype=np.float32)
    
    def _process_keypoints(self, keypoints_data):
        """Process MediaPipe keypoints to standardized format (unchanged)"""
        try:
            if isinstance(keypoints_data, list):
                keypoints = []
                for frame_data in keypoints_data:
                    if isinstance(frame_data, dict) and 'landmarks' in frame_data:
                        landmarks = frame_data['landmarks']
                        frame_kp = []
                        
                        for i in range(33):
                            if i < len(landmarks):
                                landmark = landmarks[i]
                                if isinstance(landmark, dict):
                                    x = float(landmark.get('x', 0.0))
                                    y = float(landmark.get('y', 0.0))
                                    z = float(landmark.get('z', 0.0))
                                    frame_kp.append([x, y, z])
                                else:
                                    frame_kp.append([0.0, 0.0, 0.0])
                            else:
                                frame_kp.append([0.0, 0.0, 0.0])
                        keypoints.append(frame_kp)
                    else:
                        # Handle other formats
                        frame_kp = [[0.0, 0.0, 0.0] for _ in range(33)]
                        keypoints.append(frame_kp)
                
                return np.array(keypoints, dtype=np.float32)
            else:
                return np.zeros((60, 33, 3), dtype=np.float32)
        except Exception:
            return np.zeros((60, 33, 3), dtype=np.float32)

# Updated collate function (unchanged)
def collate_multilabel_sequences(batch):
    """Custom collate function for multi-label variable-length sequences"""
    video_names = [item['video_name'] for item in batch]
    keypoints_list = [item['keypoints'] for item in batch]
    multilabel_targets = torch.stack([item['multilabel_target'] for item in batch])
    rep_features = torch.stack([item['rep_features'] for item in batch])
    sequence_lengths = torch.stack([item['sequence_length'] for item in batch])
    
    # Pad sequences
    padded_keypoints = pad_sequence(keypoints_list, batch_first=True, padding_value=0.0)
    
    return {
        'video_names': video_names,
        'keypoints': padded_keypoints,
        'multilabel_targets': multilabel_targets,
        'rep_features': rep_features,  # Now 9D instead of 7D
        'sequence_lengths': sequence_lengths.squeeze(1)
    }

def create_enhanced_multilabel_dataloaders():
    """Create enhanced multi-label dataloaders with integrated depth features"""
    
    print("📁 Loading user-level splits for enhanced dataset...")
    splits_path = "data/squat_processed/user_level_multilabel_splits.json"
    with open(splits_path, 'r') as f:
        splits_data = json.load(f)
    
    # Create enhanced datasets
    train_dataset = EnhancedMultiLabelSquatDataset(
        video_names=splits_data['splits']['train']['video_names'],
        multilabel_targets=splits_data['splits']['train']['multilabel_targets'],
        enable_depth_integration=True,
        enable_clean_biomech=True
    )
    
    val_dataset = EnhancedMultiLabelSquatDataset(
        video_names=splits_data['splits']['validation']['video_names'],
        multilabel_targets=splits_data['splits']['validation']['multilabel_targets'],
        enable_depth_integration=True,
        enable_clean_biomech=True
    )
    
    test_dataset = EnhancedMultiLabelSquatDataset(
        video_names=splits_data['splits']['test']['video_names'],
        multilabel_targets=splits_data['splits']['test']['multilabel_targets'],
        enable_depth_integration=True,
        enable_clean_biomech=True
    )
    
    # Create dataloaders
    batch_size = 16
    
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        collate_fn=collate_multilabel_sequences, drop_last=True
    )
    
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_multilabel_sequences
    )
    
    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        collate_fn=collate_multilabel_sequences
    )
    
    print(f"✅ Created enhanced dataloaders:")
    print(f"   Train: {len(train_loader)} batches")
    print(f"   Val: {len(val_loader)} batches")
    print(f"   Test: {len(test_loader)} batches")
    print(f"   Feature dimension: 9D [5D_biomech + 2D_depth_signals + 2D_depth_rules]")
    
    return train_loader, val_loader, test_loader

# Create enhanced dataloaders
train_loader, val_loader, test_loader = create_enhanced_multilabel_dataloaders()
print("\n🎉 ENHANCED MULTI-LABEL DATALOADERS WITH DEPTH INTEGRATION READY!")

INFO:cleaned_biomech_extractor:CleanedSquatBiomechanicalExtractor initialized (5D features)
INFO:cleaned_biomech_extractor:CleanedSquatBiomechanicalExtractor initialized (5D features)
INFO:cleaned_biomech_extractor:CleanedSquatBiomechanicalExtractor initialized (5D features)


📁 Loading user-level splits for enhanced dataset...
✅ Loaded depth thresholds: INDIVIDUAL_hip_knee_ratio_bottom
✅ Loaded cleaned 5D biomechanical feature extractor
Dataset: 1123/1137 valid videos
   Depth integration: ENABLED
   Clean biomech: ENABLED
✅ Loaded depth thresholds: INDIVIDUAL_hip_knee_ratio_bottom
✅ Loaded cleaned 5D biomechanical feature extractor
Dataset: 242/244 valid videos
   Depth integration: ENABLED
   Clean biomech: ENABLED
✅ Loaded depth thresholds: INDIVIDUAL_hip_knee_ratio_bottom
✅ Loaded cleaned 5D biomechanical feature extractor
Dataset: 242/244 valid videos
   Depth integration: ENABLED
   Clean biomech: ENABLED
✅ Created enhanced dataloaders:
   Train: 70 batches
   Val: 16 batches
   Test: 16 batches
   Feature dimension: 9D [5D_biomech + 2D_depth_signals + 2D_depth_rules]

🎉 ENHANCED MULTI-LABEL DATALOADERS WITH DEPTH INTEGRATION READY!


In [16]:
# FIXED CELL 4: Multi-Label CNN-LSTM Architecture with 9D Features + Factory Function Fix
# REPLACE your existing Cell 4 with this code

class TemporalConvBlock(nn.Module):
    """1D Temporal convolution block"""
    
    def __init__(self, in_channels, out_channels, kernel_size=5, dropout=0.1):
        super().__init__()
        
        self.conv1 = nn.Conv1d(in_channels, out_channels, kernel_size, 
                              padding=kernel_size//2, bias=False)
        self.bn1 = nn.BatchNorm1d(out_channels)
        self.conv2 = nn.Conv1d(out_channels, out_channels, kernel_size,
                              padding=kernel_size//2, bias=False)
        self.bn2 = nn.BatchNorm1d(out_channels)

        if in_channels != out_channels:
            self.residual = nn.Conv1d(in_channels, out_channels, 1, bias=False) 
        else:
            self.residual = nn.Identity()
        
        self.dropout = nn.Dropout(dropout)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        residual = self.residual(x)
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        out = self.dropout(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        out += residual
        out = self.relu(out)
        
        return out

class MultiHeadAttention(nn.Module):
    """Multi-head self-attention"""
    
    def __init__(self, embed_dim, num_heads=8, dropout=0.1):
        super().__init__()
        
        assert embed_dim % num_heads == 0
        
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3, bias=False)
        self.proj = nn.Linear(embed_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)
    
    def forward(self, x, mask=None):
        B, T, D = x.shape
        
        qkv = self.qkv(x).reshape(B, T, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        
        if mask is not None:
            mask = mask.unsqueeze(1).unsqueeze(1)
            scores = scores.masked_fill(mask, float('-inf'))
        
        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)
        
        out = torch.matmul(attn_weights, v)
        out = out.transpose(1, 2).reshape(B, T, D)
        out = self.proj(out)
        
        return out, attn_weights.mean(dim=1)

class MultiLabelHybridSquatCNNLSTM(nn.Module):
    """FIXED: Multi-Label Hybrid CNN-LSTM with 9D Features"""
    
    def __init__(self, input_dim=99, hidden_dim=256, num_lstm_layers=2, 
                 num_attention_heads=8, dropout=0.3, use_biomech_features=True):
        super().__init__()
        
        self.use_biomech_features = use_biomech_features
        
        # 1. Temporal CNN Feature Extractor
        self.temporal_cnn = nn.Sequential(
            TemporalConvBlock(input_dim, 128, kernel_size=7, dropout=dropout),
            TemporalConvBlock(128, 256, kernel_size=5, dropout=dropout),
            TemporalConvBlock(256, 256, kernel_size=3, dropout=dropout)
        )
        
        # 2. Bidirectional LSTM
        self.lstm = nn.LSTM(
            input_size=256, hidden_size=hidden_dim, num_layers=num_lstm_layers,
            bidirectional=True, batch_first=True,
            dropout=dropout if num_lstm_layers > 1 else 0
        )
        
        lstm_output_dim = hidden_dim * 2
        
        # 3. Multi-head Attention
        self.attention = MultiHeadAttention(lstm_output_dim, num_attention_heads, dropout)
        
        # 4. Global Context Extraction
        self.context_layer = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 1)
        )
        
        # 5. FIXED: 9D Biomechanical Feature Projection
        if self.use_biomech_features:
            self.rep_proj = nn.Sequential(
                nn.Linear(9, 64),  # FIXED: 9D features
                nn.ReLU(),
                nn.Dropout(dropout * 0.5)
            )
            fusion_input_dim = lstm_output_dim + 64  # 512 + 64 = 576
        else:
            self.rep_proj = None
            fusion_input_dim = lstm_output_dim  # 512
        
        # 6. Fusion Head
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # 7. Multi-Label Output Heads
        self.good_form_head = nn.Linear(256, 1)      # Good vs not good
        self.posture_fault_head = nn.Linear(256, 1)  # Posture fault yes/no
        self.depth_fault_head = nn.Linear(256, 1)    # Depth fault yes/no
        
        self._init_weights()
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"✅ MultiLabelHybridSquatCNNLSTM initialized:")
        print(f"   Parameters: {total_params:,}")
        print(f"   Biomech features: {'ENABLED (9D)' if use_biomech_features else 'DISABLED'}")
        print(f"   Feature composition: [5D_biomech + 2D_depth_signals + 2D_depth_rules]")
        print(f"   Fusion: CONCATENATION (512 + 64 = 576)")
        print(f"   Output: Multi-label (3 logits)")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0)
    
    def forward(self, keypoints, sequence_lengths, rep_features=None, return_attention=False):
        batch_size, max_seq_len = keypoints.size(0), keypoints.size(1)
        
        # Flatten keypoints: (B, T, 33, 3) → (B, T, 99)
        x = keypoints.view(batch_size, max_seq_len, -1)
        
        # 1. Temporal CNN
        x = x.transpose(1, 2)  # (B, 99, T)
        x = self.temporal_cnn(x)  # (B, 256, T)
        x = x.transpose(1, 2)  # (B, T, 256)
        
        # 2. LSTM with packing
        packed_x = pack_padded_sequence(x, sequence_lengths.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed_x)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)  # (B, T, 512)
        
        # 3. Multi-head Attention
        mask = self._create_padding_mask(sequence_lengths, max_seq_len).to(keypoints.device)
        attn_out, attn_weights = self.attention(lstm_out, mask)
        
        # 4. Global Context Pooling
        context_scores = self.context_layer(attn_out)
        context_scores = context_scores.masked_fill(mask.unsqueeze(-1), float('-inf'))
        context_weights = F.softmax(context_scores.squeeze(-1), dim=1)
        sequence_repr = torch.sum(attn_out * context_weights.unsqueeze(-1), dim=1)  # (B, 512)
        
        # 5. FIXED: 9D Feature Fusion
        if self.use_biomech_features and rep_features is not None:
            # STRICT 9D validation
            if rep_features.size(-1) != 9:
                raise ValueError(f"Expected exactly 9D features, got {rep_features.size(-1)}D. Fix dataset!")
            
            rep_proj = self.rep_proj(rep_features)  # (B, 64)
            fused_features = torch.cat([sequence_repr, rep_proj], dim=-1)  # (B, 576)
        else:
            fused_features = sequence_repr  # (B, 512)
        
        # 6. Fusion Head
        x = self.fusion_head(fused_features)  # (B, 256)
        
        # 7. Multi-Label Outputs (raw logits for BCEWithLogitsLoss)
        good_logit = self.good_form_head(x)        # (B, 1)
        posture_logit = self.posture_fault_head(x) # (B, 1)
        depth_logit = self.depth_fault_head(x)     # (B, 1)
        
        logits = torch.cat([good_logit, posture_logit, depth_logit], dim=1)  # (B, 3)
        
        if return_attention:
            return logits, {
                'self_attention': attn_weights,
                'context_attention': context_weights,
                'sequence_repr': sequence_repr,
                'fused_features': fused_features
            }
        else:
            return logits
    
    def _create_padding_mask(self, sequence_lengths, max_seq_len):
        batch_size = sequence_lengths.size(0)
        mask = torch.arange(max_seq_len).expand(batch_size, max_seq_len) >= sequence_lengths.unsqueeze(1)
        return mask

class MultiLabelHybridSquatCNNLSTM_Small(nn.Module):
    """FIXED: Small variant with 9D features"""
    
    def __init__(self, input_dim=99, hidden_dim=128, num_lstm_layers=1, 
                 num_attention_heads=8, dropout=0.3, use_biomech_features=True):
        super().__init__()
        
        self.use_biomech_features = use_biomech_features
        
        # Same CNN
        self.temporal_cnn = nn.Sequential(
            TemporalConvBlock(input_dim, 128, kernel_size=7, dropout=dropout),
            TemporalConvBlock(128, 256, kernel_size=5, dropout=dropout),
            TemporalConvBlock(256, 256, kernel_size=3, dropout=dropout)
        )
        
        # Smaller LSTM: 1 layer, hidden_dim=128
        self.lstm = nn.LSTM(
            input_size=256, hidden_size=hidden_dim, num_layers=num_lstm_layers,
            bidirectional=True, batch_first=True, dropout=0
        )
        
        lstm_output_dim = hidden_dim * 2  # 256
        
        # Same attention
        self.attention = MultiHeadAttention(lstm_output_dim, num_attention_heads, dropout)
        
        # Smaller context layer
        self.context_layer = nn.Sequential(
            nn.Linear(lstm_output_dim, hidden_dim//2),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim//2, 1)
        )
        
        # FIXED: 9D biomech projection
        if self.use_biomech_features:
            self.rep_proj = nn.Sequential(
                nn.Linear(9, 64),  # FIXED: 9D features
                nn.ReLU(),
                nn.Dropout(dropout * 0.5)
            )
            fusion_input_dim = lstm_output_dim + 64  # 320
        else:
            self.rep_proj = None
            fusion_input_dim = lstm_output_dim  # 256
        
        # Smaller fusion head
        self.fusion_head = nn.Sequential(
            nn.Linear(fusion_input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        
        # Same output heads
        self.good_form_head = nn.Linear(128, 1)
        self.posture_fault_head = nn.Linear(128, 1)
        self.depth_fault_head = nn.Linear(128, 1)
        
        self._init_weights()
        
        total_params = sum(p.numel() for p in self.parameters())
        print(f"✅ Small variant: {total_params:,} parameters")
        print(f"   Biomech features: 9D [5D_biomech + 2D_signals + 2D_rules]")
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.LSTM):
                for name, param in m.named_parameters():
                    if 'weight' in name:
                        nn.init.orthogonal_(param)
                    elif 'bias' in name:
                        nn.init.constant_(param, 0)
    
    def forward(self, keypoints, sequence_lengths, rep_features=None, return_attention=False):
        batch_size, max_seq_len = keypoints.size(0), keypoints.size(1)
        x = keypoints.view(batch_size, max_seq_len, -1)
        
        x = x.transpose(1, 2)
        x = self.temporal_cnn(x)
        x = x.transpose(1, 2)
        
        packed_x = pack_padded_sequence(x, sequence_lengths.cpu(), batch_first=True, enforce_sorted=False)
        lstm_out, _ = self.lstm(packed_x)
        lstm_out, _ = pad_packed_sequence(lstm_out, batch_first=True)
        
        mask = self._create_padding_mask(sequence_lengths, max_seq_len).to(keypoints.device)
        attn_out, attn_weights = self.attention(lstm_out, mask)
        
        context_scores = self.context_layer(attn_out)
        context_scores = context_scores.masked_fill(mask.unsqueeze(-1), float('-inf'))
        context_weights = F.softmax(context_scores.squeeze(-1), dim=1)
        sequence_repr = torch.sum(attn_out * context_weights.unsqueeze(-1), dim=1)
        
        if self.use_biomech_features and rep_features is not None:
            # STRICT 9D validation
            if rep_features.size(-1) != 9:
                raise ValueError(f"Expected exactly 9D features, got {rep_features.size(-1)}D. Fix dataset!")
            
            rep_proj = self.rep_proj(rep_features)
            fused_features = torch.cat([sequence_repr, rep_proj], dim=-1)
        else:
            fused_features = sequence_repr
        
        x = self.fusion_head(fused_features)
        
        good_logit = self.good_form_head(x)
        posture_logit = self.posture_fault_head(x)
        depth_logit = self.depth_fault_head(x)
        
        logits = torch.cat([good_logit, posture_logit, depth_logit], dim=1)
        
        if return_attention:
            return logits, {
                'self_attention': attn_weights,
                'context_attention': context_weights,
                'sequence_repr': sequence_repr,
                'fused_features': fused_features
            }
        else:
            return logits
    
    def _create_padding_mask(self, sequence_lengths, max_seq_len):
        batch_size = sequence_lengths.size(0)
        mask = torch.arange(max_seq_len).expand(batch_size, max_seq_len) >= sequence_lengths.unsqueeze(1)
        return mask

# FIXED: Factory function with backwards compatibility
def create_enhanced_model(model_type='original', **kwargs):
    """Create model based on type"""
    if model_type == 'original':
        return MultiLabelHybridSquatCNNLSTM(**kwargs)
    elif model_type == 'small':
        return MultiLabelHybridSquatCNNLSTM_Small(**kwargs)
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

# BACKWARDS COMPATIBILITY: Alias for old code
def create_model(model_type='original', **kwargs):
    """Backwards compatibility alias"""
    return create_enhanced_model(model_type=model_type, **kwargs)

# CELL ORDER INDEPENDENCE: Test only if dataloaders exist
if 'train_loader' in globals():
    print("🧪 Testing 9D architecture with real data...")
    model = create_enhanced_model(model_type='original', use_biomech_features=True).to(device)
    
    # Test forward pass with first batch
    batch = next(iter(train_loader))
    keypoints = batch['keypoints'].to(device)
    rep_features = batch['rep_features'].to(device)
    sequence_lengths = batch['sequence_lengths'].to(device)
    
    print(f"\n📊 9D feature validation:")
    print(f"   Input keypoints: {keypoints.shape}")
    print(f"   Input features: {rep_features.shape} (should be [B, 9])")
    print(f"   Feature sample: {rep_features[0].cpu().numpy()[:5]}... (first 5 values)")
    
    with torch.no_grad():
        logits = model(keypoints, sequence_lengths, rep_features)
    
    print(f"\n✅ Forward pass successful:")
    print(f"   Output: {logits.shape} (batch_size, 3)")
    print(f"   Logits range: [{logits.min().item():.2f}, {logits.max().item():.2f}]")
    
    del model, batch  # Clean up
else:
    print("⏭️  train_loader not defined yet — skipping forward pass test")
    print("   (This is normal if running Cell 4 before dataloaders)")

print("\n🎉 FIXED 9D ARCHITECTURE READY!")
print("✅ Fixes applied:")
print("   - create_model() NameError → Fixed with backwards compatibility")
print("   - 9D feature validation → Strict enforcement")
print("   - Cell order independence → No train_loader dependency")
print("   - Proper Linear(9, 64) dimensions for both models")

🧪 Testing 9D architecture with real data...
✅ MultiLabelHybridSquatCNNLSTM initialized:
   Parameters: 5,096,068
   Biomech features: ENABLED (9D)
   Feature composition: [5D_biomech + 2D_depth_signals + 2D_depth_rules]
   Fusion: CONCATENATION (512 + 64 = 576)
   Output: Multi-label (3 logits)

📊 9D feature validation:
   Input keypoints: torch.Size([16, 208, 33, 3])
   Input features: torch.Size([16, 9]) (should be [B, 9])
   Feature sample: [83.41961    94.15731    89.43584    14.597994    0.11354182]... (first 5 values)

✅ Forward pass successful:
   Output: torch.Size([16, 3]) (batch_size, 3)
   Logits range: [-2.53, 3.32]

🎉 FIXED 9D ARCHITECTURE READY!
✅ Fixes applied:
   - create_model() NameError → Fixed with backwards compatibility
   - 9D feature validation → Strict enforcement
   - Cell order independence → No train_loader dependency
   - Proper Linear(9, 64) dimensions for both models


In [18]:
# FIXED REPLACEMENT CELL 5 — Training Loop with Best Checkpoint Saving
# Replace the existing Cell 5 in 06_multilabel_CNN_LSTM_refactored.ipynb with this code

import copy
from pathlib import Path
import json
from datetime import datetime

def safe_copy_optimizer_state(optimizer_state_dict):
    """Safely copy optimizer state dict to CPU, handling mixed types"""
    safe_state = {}
    for key, value in optimizer_state_dict.items():
        if hasattr(value, 'cpu') and hasattr(value, 'clone'):
            # It's a tensor, move to CPU and clone
            safe_state[key] = value.cpu().clone()
        elif isinstance(value, dict):
            # It's a nested dict (like 'state'), recursively handle
            safe_state[key] = {}
            for sub_key, sub_value in value.items():
                if hasattr(sub_value, 'cpu') and hasattr(sub_value, 'clone'):
                    safe_state[key][sub_key] = sub_value.cpu().clone()
                else:
                    safe_state[key][sub_key] = copy.deepcopy(sub_value)
        else:
            # It's a regular value (int, float, list, etc.), deep copy
            safe_state[key] = copy.deepcopy(value)
    return safe_state

class MultiLabelTrainer:
    """Multi-label trainer with robust checkpoint saving"""
    
    def __init__(self, model, device, pos_weights=None, run_dir="runs/exp_small_variant_training_checkpoints"):
        self.model = model.to(device)
        self.device = device
        self.run_dir = Path(run_dir)
        self.run_dir.mkdir(parents=True, exist_ok=True)
        
        if pos_weights is not None:
            self.criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights.to(device))
            self.pos_weights = pos_weights.cpu()  # Store for checkpointing
        else:
            self.criterion = nn.BCEWithLogitsLoss()
            self.pos_weights = None
        
        self.train_metrics = MultiLabelMetrics()
        self.val_metrics = MultiLabelMetrics()
        
        self.history = {
            'train_loss': [], 'val_loss': [],
            'train_macro_f1': [], 'val_macro_f1': [],
            'train_exact_match': [], 'val_exact_match': [],
            'learning_rates': []
        }
        
        # Checkpointing state
        self.best_val_f1 = -1.0
        self.best_epoch = -1
        
        print(f"📁 Checkpoint directory: {self.run_dir}")
    
    def _save_checkpoint(self, path, extra_data=None):
        """Save checkpoint with model state and metadata"""
        checkpoint_data = {
            'model_state_dict': {k: v.cpu().clone() for k, v in self.model.state_dict().items()},
            'epoch': extra_data.get('epoch', -1),
            'best_val_f1': self.best_val_f1,
            'MODEL_TYPE': extra_data.get('MODEL_TYPE', 'unknown'),
            'timestamp': datetime.now().isoformat(),
            'model_name': self.model.__class__.__name__
        }
        
        # Add optional data
        if self.pos_weights is not None:
            checkpoint_data['pos_weights'] = self.pos_weights.tolist()
        
        if extra_data:
            for key in ['optimizer_state_dict', 'val_metrics', 'thresholds_used']:
                if key in extra_data and extra_data[key] is not None:
                    checkpoint_data[key] = extra_data[key]
        
        # Atomic write
        temp_path = path.with_suffix('.tmp')
        torch.save(checkpoint_data, temp_path)
        temp_path.rename(path)
        
        return checkpoint_data
    
    def train_epoch(self, train_loader, optimizer):
        self.model.train()
        self.train_metrics.reset()
        
        for batch_idx, batch in enumerate(train_loader):
            keypoints = batch['keypoints'].to(self.device)
            multilabel_targets = batch['multilabel_targets'].to(self.device)
            rep_features = batch['rep_features'].to(self.device)
            sequence_lengths = batch['sequence_lengths'].to(self.device)
            
            optimizer.zero_grad()
            
            try:
                logits = self.model(keypoints, sequence_lengths, rep_features)
                loss = self.criterion(logits, multilabel_targets)
                
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
                optimizer.step()
                
                self.train_metrics.update(logits, multilabel_targets, loss)
                
                if (batch_idx + 1) % 10 == 0:
                    print(f"   Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
            except Exception as e:
                print(f"   Skipping batch {batch_idx+1}: {e}")
                continue
        
        return self.train_metrics.compute()
    
    def validate_epoch(self, val_loader):
        self.model.eval()
        self.val_metrics.reset()
        
        with torch.no_grad():
            for batch in val_loader:
                keypoints = batch['keypoints'].to(self.device)
                multilabel_targets = batch['multilabel_targets'].to(self.device)
                rep_features = batch['rep_features'].to(self.device)
                sequence_lengths = batch['sequence_lengths'].to(self.device)
                
                try:
                    logits = self.model(keypoints, sequence_lengths, rep_features)
                    loss = self.criterion(logits, multilabel_targets)
                    self.val_metrics.update(logits, multilabel_targets, loss)
                except Exception as e:
                    continue
        
        return self.val_metrics.compute()
    
    def fit(self, train_loader, val_loader, num_epochs=50, learning_rate=0.001, patience=10, model_type='unknown'):
        print(f"🚀 MULTI-LABEL TRAINING WITH CHECKPOINTING:")
        print(f"   Epochs: {num_epochs} | LR: {learning_rate} | Patience: {patience}")
        print(f"   Model type: {model_type}")
        print(f"   Checkpoint dir: {self.run_dir}")
        
        optimizer = optim.AdamW(self.model.parameters(), lr=learning_rate, weight_decay=0.01)
        scheduler = ReduceLROnPlateau(optimizer, mode='max', patience=patience//2, factor=0.5, verbose=True)
        
        patience_counter = 0
        
        for epoch in range(num_epochs):
            print(f"\n📈 EPOCH {epoch+1}/{num_epochs}")
            
            train_metrics = self.train_epoch(train_loader, optimizer)
            val_metrics = self.validate_epoch(val_loader)
            
            scheduler.step(val_metrics['overall']['macro_f1'])
            current_lr = optimizer.param_groups[0]['lr']
            
            # Store history
            self.history['train_loss'].append(train_metrics['loss'])
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['train_macro_f1'].append(train_metrics['overall']['macro_f1'])
            self.history['val_macro_f1'].append(val_metrics['overall']['macro_f1'])
            self.history['train_exact_match'].append(train_metrics['overall']['exact_match_accuracy'])
            self.history['val_exact_match'].append(val_metrics['overall']['exact_match_accuracy'])
            self.history['learning_rates'].append(current_lr)
            
            # Progress reporting
            print(f"📊 Results: Train Loss={train_metrics['loss']:.4f}, Val Loss={val_metrics['loss']:.4f}")
            print(f"   Macro F1: Train={train_metrics['overall']['macro_f1']:.4f}, Val={val_metrics['overall']['macro_f1']:.4f}")
            print(f"   Exact Match: Train={train_metrics['overall']['exact_match_accuracy']:.4f}, Val={val_metrics['overall']['exact_match_accuracy']:.4f}")
            
            # Per-head metrics
            print(f"   Val Per-Head F1:")
            for head_name in ['good_form', 'posture_fault', 'depth_fault']:
                f1 = val_metrics['per_head'][head_name]['f1']
                precision = val_metrics['per_head'][head_name]['precision']
                recall = val_metrics['per_head'][head_name]['recall']
                print(f"     {head_name}: F1={f1:.3f}, P={precision:.3f}, R={recall:.3f}")
            
            # Save rolling checkpoint every epoch
            last_checkpoint_path = self.run_dir / f"last_model__{model_type}.pt"
            self._save_checkpoint(last_checkpoint_path, {
                'epoch': epoch,
                'MODEL_TYPE': model_type,
                'optimizer_state_dict': safe_copy_optimizer_state(optimizer.state_dict()),  # FIXED
                'val_metrics': val_metrics
            })
            
            # Check for new best model
            val_f1 = val_metrics['overall']['macro_f1']
            if val_f1 > self.best_val_f1:
                self.best_val_f1 = val_f1
                self.best_epoch = epoch
                patience_counter = 0
                
                # Save best checkpoint immediately
                best_checkpoint_path = self.run_dir / f"best_model__{model_type}.pt"
                checkpoint_data = self._save_checkpoint(best_checkpoint_path, {
                    'epoch': epoch,
                    'MODEL_TYPE': model_type,
                    'optimizer_state_dict': safe_copy_optimizer_state(optimizer.state_dict()),  # FIXED
                    'val_metrics': val_metrics,
                    'thresholds_used': None  # No thresholds during training
                })
                
                print(f"   ⭐ NEW BEST MODEL! Macro F1: {self.best_val_f1:.4f} (epoch {epoch+1})")
                print(f"   💾 Saved: {best_checkpoint_path}")
            else:
                patience_counter += 1
            
            # Early stopping
            if patience_counter >= patience:
                print(f"\n⏹️ Early stopping at epoch {epoch+1}")
                break
        
        print(f"\n🎉 Training completed!")
        print(f"   Best validation macro F1: {self.best_val_f1:.4f} (epoch {self.best_epoch+1})")
        print(f"   Best checkpoint: {self.run_dir}/best_model__{model_type}.pt")
        print(f"   Last checkpoint: {self.run_dir}/last_model__{model_type}.pt")
        
        return self.history

# Compute class weights (same as original)
print("⚖️ Computing positive class weights...")
pos_counts = torch.zeros(3)
total_count = 0

for batch in train_loader:
    targets = batch['multilabel_targets']
    pos_counts += targets.sum(dim=0)
    total_count += targets.size(0)

neg_counts = total_count - pos_counts
pos_weights = neg_counts / pos_counts
print(f"Positive weights: {pos_weights.numpy()}")

# Create model based on MODEL_TYPE (assumes MODEL_TYPE is defined in original notebook)
print(f"\n🏗️ Using {MODEL_TYPE} model variant")
model = create_model(model_type=MODEL_TYPE, use_biomech_features=True).to(device)

# Show parameter comparison if small variant
if MODEL_TYPE == 'small':
    orig_model = create_model('original', use_biomech_features=True)
    orig_params = sum(p.numel() for p in orig_model.parameters())
    small_params = sum(p.numel() for p in model.parameters())
    reduction = (1 - small_params / orig_params) * 100
    print(f"📊 Parameter reduction: {reduction:.1f}% ({orig_params:,} → {small_params:,})")
    del orig_model

# Create trainer with checkpointing
trainer = MultiLabelTrainer(model, device, pos_weights=pos_weights)

# Run training with checkpointing
history = trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=50,
    learning_rate=0.001,
    patience=8,
    model_type=MODEL_TYPE
)

print("\n🎉 MULTI-LABEL TRAINING WITH CHECKPOINTING COMPLETE!")

⚖️ Computing positive class weights...
Positive weights: [1.8211586 1.835443  2.4146342]

🏗️ Using small model variant
✅ Small variant: 1,852,676 parameters
   Biomech features: 9D [5D_biomech + 2D_signals + 2D_rules]
✅ MultiLabelHybridSquatCNNLSTM initialized:
   Parameters: 5,096,068
   Biomech features: ENABLED (9D)
   Feature composition: [5D_biomech + 2D_depth_signals + 2D_depth_rules]
   Fusion: CONCATENATION (512 + 64 = 576)
   Output: Multi-label (3 logits)
📊 Parameter reduction: 63.6% (5,096,068 → 1,852,676)
📁 Checkpoint directory: runs/exp_small_variant_training_checkpoints
🚀 MULTI-LABEL TRAINING WITH CHECKPOINTING:
   Epochs: 50 | LR: 0.001 | Patience: 8
   Model type: small
   Checkpoint dir: runs/exp_small_variant_training_checkpoints

📈 EPOCH 1/50
   Batch 10/70 | Loss: 1.0437
   Batch 20/70 | Loss: 0.8605
   Batch 30/70 | Loss: 1.3523
   Batch 40/70 | Loss: 1.1195
   Batch 50/70 | Loss: 0.8811
   Batch 60/70 | Loss: 0.9577
   Batch 70/70 | Loss: 0.9907
📊 Results: Train L

In [19]:
# NEW CELL 6 — Post-Training: Export Best Checkpoint, Metrics, and Outputs (NPZ)
# Add this as a new cell after training completes in the original notebook

import numpy as np
from pathlib import Path
import json
from datetime import datetime

def collect_outputs(model, loader, device):
    """
    Collect model outputs (logits, probs, targets, video_names)
    
    Returns:
        dict with numpy arrays: logits, probs, targets, video_names
    """
    model.eval()
    all_logits = []
    all_targets = []
    all_video_names = []
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            try:
                keypoints = batch['keypoints'].to(device)
                targets = batch['multilabel_targets'].to(device)
                rep_features = batch['rep_features'].to(device)
                sequence_lengths = batch['sequence_lengths'].to(device)
                
                # Get video names if available, otherwise create sequential IDs
                if 'video_names' in batch:
                    video_names = batch['video_names']
                else:
                    video_names = [f"sample_{batch_idx}_{i}" for i in range(len(targets))]
                
                # Forward pass
                logits = model(keypoints, sequence_lengths, rep_features)
                
                # Store outputs
                all_logits.append(logits.cpu())
                all_targets.append(targets.cpu())
                all_video_names.extend(video_names)
                
            except Exception as e:
                print(f"   ⚠️  Skipping batch {batch_idx + 1}: {e}")
                continue
    
    if not all_logits:
        raise ValueError("No valid batches processed!")
    
    # Concatenate and convert to numpy
    logits = torch.cat(all_logits, dim=0).numpy()  # (N, 3)
    targets = torch.cat(all_targets, dim=0).numpy()  # (N, 3)
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid to get probabilities
    
    # Validate outputs
    assert np.isfinite(logits).all(), "Non-finite logits detected!"
    assert np.isfinite(probs).all(), "Non-finite probabilities detected!"
    assert len(all_video_names) == len(logits), "Video names count mismatch!"
    
    return {
        'logits': logits,
        'probs': probs,
        'targets': targets,
        'video_names': all_video_names
    }

print(f"📤 POST-TRAINING: Exporting Best Checkpoint Outputs")

# Locate best checkpoint
checkpoint_dir = trainer.run_dir
best_checkpoint_path = checkpoint_dir / f"best_model__{MODEL_TYPE}.pt"

if not best_checkpoint_path.exists():
    raise FileNotFoundError(f"Best checkpoint not found: {best_checkpoint_path}")

print(f"📁 Loading best checkpoint: {best_checkpoint_path}")

# Load best checkpoint
checkpoint = torch.load(best_checkpoint_path, map_location=device)

# Restore model to best state
trainer.model.load_state_dict(checkpoint['model_state_dict'], strict=True)
trainer.model.eval()

print(f"✅ Restored model to best state:")
print(f"   Epoch: {checkpoint['epoch'] + 1}")
print(f"   Best val macro F1: {checkpoint['best_val_f1']:.4f}")
print(f"   Model type: {checkpoint['MODEL_TYPE']}")

# Collect validation outputs
print(f"\n📊 Collecting validation outputs...")
val_outputs = collect_outputs(trainer.model, val_loader, device)
print(f"   Validation samples: {len(val_outputs['logits'])}")
print(f"   Shapes: logits={val_outputs['logits'].shape}, probs={val_outputs['probs'].shape}, targets={val_outputs['targets'].shape}")

# Collect test outputs
print(f"\n📊 Collecting test outputs...")
test_outputs = collect_outputs(trainer.model, test_loader, device)
print(f"   Test samples: {len(test_outputs['logits'])}")
print(f"   Shapes: logits={test_outputs['logits'].shape}, probs={test_outputs['probs'].shape}, targets={test_outputs['targets'].shape}")

# Save validation NPZ
val_npz_path = checkpoint_dir / 'val_outputs.npz'
np.savez(
    val_npz_path,
    logits=val_outputs['logits'],
    probs=val_outputs['probs'],
    targets=val_outputs['targets'],
    video_names=val_outputs['video_names']
)
print(f"\n💾 Saved validation outputs: {val_npz_path}")

# Save test NPZ
test_npz_path = checkpoint_dir / 'test_outputs.npz'
np.savez(
    test_npz_path,
    logits=test_outputs['logits'],
    probs=test_outputs['probs'],
    targets=test_outputs['targets'],
    video_names=test_outputs['video_names']
)
print(f"💾 Saved test outputs: {test_npz_path}")

# Compute class balance (target prevalence)
head_names = ['good_form', 'posture_fault', 'depth_fault']
val_prevalence = {}
test_prevalence = {}

for i, head_name in enumerate(head_names):
    val_pos = val_outputs['targets'][:, i].sum()
    val_total = len(val_outputs['targets'])
    val_prevalence[head_name] = {'positive': int(val_pos), 'total': val_total, 'fraction': val_pos / val_total}
    
    test_pos = test_outputs['targets'][:, i].sum()
    test_total = len(test_outputs['targets'])
    test_prevalence[head_name] = {'positive': int(test_pos), 'total': test_total, 'fraction': test_pos / test_total}

# Create summary JSON
best_summary = {
    'best_checkpoint_path': str(best_checkpoint_path),
    'best_epoch': checkpoint['epoch'] + 1,  # 1-indexed for humans
    'best_val_macro_f1': checkpoint['best_val_f1'],
    'MODEL_TYPE': checkpoint['MODEL_TYPE'],
    'model_name': checkpoint.get('model_name', 'unknown'),
    'total_parameters': sum(p.numel() for p in trainer.model.parameters()),
    'class_balance': {
        'validation': val_prevalence,
        'test': test_prevalence
    },
    'output_files': {
        'val_outputs_npz': str(val_npz_path),
        'test_outputs_npz': str(test_npz_path)
    },
    'export_timestamp': datetime.now().isoformat(),
    'feature_info': {
        'feature_dim': '9D',
        'composition': '5D_biomech + 2D_depth_signals + 2D_depth_rules'
    }
}

# Save summary JSON
summary_json_path = checkpoint_dir / 'best_summary.json'
with open(summary_json_path, 'w') as f:
    json.dump(best_summary, f, indent=2)

print(f"💾 Saved best summary: {summary_json_path}")

# Print class balance summary
print(f"\n📊 Class Balance Summary:")
print(f"   Validation set ({val_outputs['logits'].shape[0]} samples):")
for head_name, stats in val_prevalence.items():
    print(f"     {head_name}: {stats['positive']}/{stats['total']} ({stats['fraction']:.3f})")

print(f"   Test set ({test_outputs['logits'].shape[0]} samples):")
for head_name, stats in test_prevalence.items():
    print(f"     {head_name}: {stats['positive']}/{stats['total']} ({stats['fraction']:.3f})")

print(f"\n🎉 EXPORT COMPLETE!")
print(f"📁 All files saved to: {checkpoint_dir}")
print(f"   - best_model__{MODEL_TYPE}.pt (checkpoint)")
print(f"   - val_outputs.npz (validation logits/probs/targets)")
print(f"   - test_outputs.npz (test logits/probs/targets)")
print(f"   - best_summary.json (metadata)")
print(f"\n🚀 Ready for threshold optimization and deployment!")

📤 POST-TRAINING: Exporting Best Checkpoint Outputs
📁 Loading best checkpoint: runs/exp_small_variant_training_checkpoints/best_model__small.pt
✅ Restored model to best state:
   Epoch: 15
   Best val macro F1: 0.5606
   Model type: small

📊 Collecting validation outputs...
   Validation samples: 242
   Shapes: logits=(242, 3), probs=(242, 3), targets=(242, 3)

📊 Collecting test outputs...
   Test samples: 242
   Shapes: logits=(242, 3), probs=(242, 3), targets=(242, 3)

💾 Saved validation outputs: runs/exp_small_variant_training_checkpoints/val_outputs.npz
💾 Saved test outputs: runs/exp_small_variant_training_checkpoints/test_outputs.npz
💾 Saved best summary: runs/exp_small_variant_training_checkpoints/best_summary.json

📊 Class Balance Summary:
   Validation set (242 samples):
     good_form: 86/242 (0.355)
     posture_fault: 87/242 (0.360)
     depth_fault: 69/242 (0.285)
   Test set (242 samples):
     good_form: 87/242 (0.360)
     posture_fault: 85/242 (0.351)
     depth_fault: 7